# 5 · make — training-data composition

What the model behind every other figure was trained on, by source. Two numbers per source, both
of record and from independent sources:

- **structures** — how many documents survived
  [#225](https://github.com/Open-Athena/MarinFold/issues/225)'s FoldBench decontamination, read
  from each corpus's published `build.provenance.json`;
- **tokens** — what they tokenize to, read from the launch constants
  [#232](https://github.com/Open-Athena/MarinFold/issues/232) verified its caches against before
  every training run.

The two have to agree on the document count, and this notebook stops if they do not.

`m2`, the mixture the checkpoint used, samples the corpora **in proportion to their token
counts** — so the token column is also the sampling weight, and one bar per source says both what
the corpus is made of and what the model saw.

CPU only; the bucket read is anonymous.

In [ ]:
# Run from anywhere: figlib lives next to this notebook.
import sys
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "figlib.py").exists() else Path("experiments/exp250_evals_exploration_notebook/figures")
sys.path.insert(0, str(HERE.resolve()))
import figlib

In [ ]:
# --- parameters -------------------------------------------------------------------------------
DATASET = "5_training_composition"
CORPORA = f"{figlib.BUCKET}/data/document_structures"
# source key -> where its published provenance is, what the panel calls it, and the prefix of the
# pinned token/document constants in exp232's training contract.
SOURCES = {
    "afdb": dict(
        provenance=f"{CORPORA}/contacts_v1_decontam/train/build.provenance.json",
        label="AFDB", constants="AFDB",
        description="AlphaFold DB predicted monomers (#53), decontaminated in #225"),
    "esm_atlas": dict(
        provenance=f"{CORPORA}/contacts_v1_esm_atlas_decontam/train/build.provenance.json",
        label="ESM Atlas", constants="ESM",
        description="ESM Atlas predicted metagenomic structures (#139), decontaminated in #225"),
}
# The launch constants live in the exp232 experiment. main later split that directory up, so both
# names are accepted and the one actually read is recorded in the metadata.
CONTRACT_CANDIDATES = [
    "experiments/exp232_sweep_cv1_decontam/training_contract.py",
    "experiments/exp232_sweep_cv1_decontam/exp232_sweep.py",
]
MODEL = "contacts-v1-exp232-m2-p06-train-1.5B"   # the checkpoint this corpus produced
PARAMETERS = dict(sources=list(SOURCES), contract_candidates=CONTRACT_CANDIDATES, model=MODEL)
PARAMETERS

In [ ]:
import json
import re

import pandas as pd

inputs = figlib.Inputs()
contract = next((figlib.REPO / name for name in CONTRACT_CANDIDATES
                 if (figlib.REPO / name).exists()), None)
if contract is None:
    raise SystemExit("no exp232 training contract found; looked for "
                     + ", ".join(CONTRACT_CANDIDATES))
inputs.add_file(contract)
contract_text = contract.read_text()


def constant(name: str) -> int:
    """One integer literal from the pinned contract, or a clear failure."""
    match = re.search(rf"^{name}\s*=\s*([0-9_]+)\s*$", contract_text, re.MULTILINE)
    if match is None:
        raise SystemExit(f"{name} is no longer a plain literal in {contract.name}; "
                         "exp232's training contract changed and this notebook must follow it")
    return int(match.group(1).replace("_", ""))


print(f"launch constants: {contract.relative_to(figlib.REPO)}")

In [ ]:
rows = []
for key, spec in SOURCES.items():
    published = json.loads(inputs.fetch(spec["provenance"]))
    documents = constant(f"{spec['constants']}_DOCUMENTS")
    tokens = constant(f"{spec['constants']}_TOKENS")
    # Two independent counts of the same corpus: the decontamination job counted the rows it
    # wrote, the tokenizer counted the documents it cached. If they disagree, one of them is not
    # describing the corpus this model trained on.
    if documents != published["n_documents_after"]:
        raise SystemExit(f"{key}: the contract says {documents:,} documents, the published "
                         f"corpus says {published['n_documents_after']:,}")
    rows.append(dict(source=key, label=spec["label"], description=spec["description"],
                     documents_before=published["n_documents_before"],
                     documents_dropped=published["n_dropped"], documents=documents,
                     tokens=tokens, decontamination_rule=published["rule"]))

sources = pd.DataFrame(rows)
sources["token_share"] = sources.tokens / sources.tokens.sum()
sources["document_share"] = sources.documents / sources.documents.sum()

corpus_tokens = int(sources.tokens.sum())
budget = constant("NUM_TRAIN_STEPS") * constant("GLOBAL_BATCH_SIZE") * constant("SEQ_LEN")
passes = budget / corpus_tokens
print(sources[["label", "documents_before", "documents_dropped", "documents", "tokens",
               "token_share"]].to_string(index=False))
print(f"\n{corpus_tokens:,} corpus tokens · {budget:,} training tokens · {passes:.4f} passes")

In [ ]:
figlib.write_dataset(
    DATASET,
    notebook="5_make_training_composition_data.ipynb",
    parameters=PARAMETERS,
    inputs=inputs,
    files={"sources.csv": lambda path: sources.to_csv(path, index=False)},
    extra={
        "model": MODEL,
        "contract": str(contract.relative_to(figlib.REPO)),
        "budget": {"corpus_tokens": corpus_tokens, "training_tokens": int(budget),
                   "passes": passes, "num_train_steps": constant("NUM_TRAIN_STEPS"),
                   "global_batch_size": constant("GLOBAL_BATCH_SIZE"),
                   "seq_len": constant("SEQ_LEN")},
        "mixture": {"key": "m2",
                    "definition": "each corpus sampled in proportion to its token count",
                    "weights": {row.source: row.token_share for row in sources.itertuples()}},
    })